## ***DATA***
---

- paper link:
    - wallbridge22_interspeech.pdf

- repo:
    - JUDGE-BENCH/data/dailydialog-acceptability at master · dmg-illc/JUDGE-BENCH · GitHub

- huggingface:
  - https://huggingface.co/datasets/mainlp/inferential_strategies

***ChatGPT description of the data:***
The dataset consists of 100 dialogue samples, each containing a context-response pair along with human-provided ratings of response acceptability. The all_score column contains multiple raw scores assigned by different annotators for each response, reflecting varying levels of perceived quality. Additionally, a binary acceptability label (target) is included, where 1 represents an acceptable response (10% of cases) and 0 an unacceptable response (90% of cases). The dataset also contains a q_id column, which appears to encode dialogue pair identifiers. The context_text and response_text fields store the conversation context and the corresponding response being evaluated. The data is primarily structured to analyze human perception of dialogue quality.

- Prompt info:

In [2]:
"annotations": [
        {
            "metric": "Sound Reasoning",
            "category": "categorical",
            "prompt": "{{ instance }} Is the model's reasoning sound, i.e. logically valid? Indicate either 'yes' or 'no'.",
            "labels_list": [
                "yes",
                "no"
            ]
        }

SyntaxError: '[' was never closed (113685013.py, line 1)

In [ ]:
!pwd

/Users/sebastianwefers/Desktop/LLM-as-a-Judge/UAT_project/notebooks/external/inferential_startegies


In [3]:
# Re-load necessary libraries after execution state reset
import pandas as pd
import json

# Reload the JSON file from the user's upload
json_path = "/Users/sebastianwefers/Desktop/LLM-as-a-Judge/UAT_project/notebooks/external/judge_bench/JUDGE-BENCH/data/inferential-strategies/inferential_strategies.json"  # Adjust the file name if needed
with open(json_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Extract instances into a DataFrame-friendly format
instances = data["instances"]

# Convert JSON instances into a DataFrame
df = pd.DataFrame([
    {
        "id": instance["id"],
        "instance_text": instance["instance"],
        "majority_human": instance["annotations"]["Sound Reasoning"]["majority_human"],
        "individual_scores": instance["annotations"]["Sound Reasoning"]["individual_human_scores"],
    }
    for instance in instances
])

print(df.shape)
df.head()

(300, 4)


,id,instance_text,majority_human,individual_scores
0,1,You will be shown the response of a language m...,no,"[no, no]"
1,2,You will be shown the response of a language m...,no,"[no, no]"
2,3,You will be shown the response of a language m...,no,"[no, no]"
3,4,You will be shown the response of a language m...,no,"[no, no]"
4,5,You will be shown the response of a language m...,no,"[no, no]"


In [4]:
df['instance_text'][0]

"You will be shown the response of a language model to a problem of propositional logic. Your task is to judge whether the model's reasoning is sound, i.e. logically valid.\nYou are first presented with the PROBLEM STATEMENT that the model has been given. Subsequently, the model's RESPONSE is shown. Indicate with a 'yes' if the model's response (in particular its rationale) is sound, and conclude 'no' if it is not.\n\n### PROBLEM STATEMENT\n\nExplain whether the conclusion logically follows from the set of statements below. Please report all your reasoning steps. \nEnd your reasoning with: Conclusion: True/False.\n\nStatements:\nEither there is a white marble in the box or else there is a black marble in the box, but not both\nEither there is a black marble in the box or else there is a pink marble in the box, but not both\nThere is a pink marble in the box if and only if there is a gray marble in the box\nConclusion: If there is a white marble in the box then there is a gray marble in

In [6]:
import re

def extract_intro_text(instance_text):
    """Extracts the introductory instructions before the problem statement."""
    match = re.search(r"^(.*?)\n\n### PROBLEM STATEMENT", instance_text, re.DOTALL)
    return match.group(1).strip() if match else None

def extract_problem_statement(instance_text):
    """Extracts the problem statement from the instance text."""
    match = re.search(r"### PROBLEM STATEMENT\n\n(.*?)\n\n### MODEL RESPONSE", instance_text, re.DOTALL)
    return match.group(1).strip() if match else None

def extract_statement(instance_text):
    """
    Extract the statement from the instance text. Starts with `Statements:` and ends before `Conclusion:`.
    """
    match = re.search(r"Statements:\n(.*?)\nConclusion:", instance_text, re.DOTALL)
    return match.group(1).strip() if match else None
    

def extract_model_response(instance_text):
    """Extracts the model's response from the instance text."""
    match = re.search(r"### MODEL RESPONSE\n\n(.*)", instance_text, re.DOTALL)
    return match.group(1).strip() if match else None

# Applying extraction to the DataFrame
df["intro_text"]        = df["instance_text"].apply(extract_intro_text)
df["problem_statement"] = df["instance_text"].apply(extract_problem_statement)
df["statement"]         = df["instance_text"].apply(extract_statement)
df["model_response"]    = df["instance_text"].apply(extract_model_response)

df.head()

,id,instance_text,majority_human,individual_scores,intro_text,problem_statement,statement,model_response
0,1,You will be shown the response of a language m...,no,"[no, no]",You will be shown the response of a language m...,Explain whether the conclusion logically follo...,Either there is a white marble in the box or e...,"Great, let's start!\n\nOkay, so I have the set..."
1,2,You will be shown the response of a language m...,no,"[no, no]",You will be shown the response of a language m...,Explain whether the conclusion logically follo...,There is a brown marble in the box if and only...,"Great, I'm ready to help! Here's my reasoning ..."
2,3,You will be shown the response of a language m...,no,"[no, no]",You will be shown the response of a language m...,Explain whether the conclusion logically follo...,There is a green marble in the box if and only...,"Okay, I'm ready to tackle this problem! Here a..."
3,4,You will be shown the response of a language m...,no,"[no, no]",You will be shown the response of a language m...,Explain whether the conclusion logically follo...,Either there is a red marble in the box or els...,"Great, I'm ready to tackle this problem! Here'..."
4,5,You will be shown the response of a language m...,no,"[no, no]",You will be shown the response of a language m...,Explain whether the conclusion logically follo...,There is a purple marble in the box if and onl...,"Okay, I'm ready to tackle this problem! Here's..."


In [10]:
idx = 1

print(f"- Problem statement:\n{df['problem_statement'][idx]}\n")
print(99*"-")
print(f"- Statement:\n{df['statement'][idx]}\n")
print(99*"-")
print(f" - Model response:\n{df['model_response'][idx]}")


- Problem statement:
Explain whether the conclusion logically follows from the set of statements below. Please report all your reasoning steps. 
End your reasoning with: Conclusion: True/False.

Statements:
There is a brown marble in the box if and only if there is an orange marble in the box
Either there is an orange marble in the box or else there is a yellow marble in the box, but not both
There is a yellow marble in the box if and only if there is a green marble in the box
Conclusion: If there is a brown marble in the box then there is a green marble in the box

Let's think step by step.

---------------------------------------------------------------------------------------------------
- Statement:
There is a brown marble in the box if and only if there is an orange marble in the box
Either there is an orange marble in the box or else there is a yellow marble in the box, but not both
There is a yellow marble in the box if and only if there is a green marble in the box

-----------

- Score distributions:

In [ ]:
df.majority_human.value_counts()

majority_human
no     242
yes     58
Name: count, dtype: int64

In [ ]:
df['human_#1'] = df['individual_scores'].apply(lambda x: x[0])
df['human_#2'] = df['individual_scores'].apply(lambda x: x[1])


In [ ]:
print(df['human_#1'].value_counts())
print()
print(df['human_#2'].value_counts())

human_#1
no     242
yes     58
Name: count, dtype: int64

human_#2
no     242
yes     58
Name: count, dtype: int64


In [ ]:
mapping = {
    "yes": 1,
    "no": 0
}

df["majority_human"] = df["majority_human"].map(mapping)
df["human_#1"] = df["human_#1"].map(mapping)
df["human_#2"] = df["human_#2"].map(mapping)

df['human_delta'] = abs(df['human_#1'] - df['human_#2'])

df['human_delta'].value_counts()

human_delta
0    300
Name: count, dtype: int64